# EDA


In [ ]:
import ast
import collections
import json

import pandas as pd
from matplotlib import pyplot as plt

from core.datasets import (
    MARKER_COLS,
    PAD_TOKEN,
    UNK_TOKEN,
    build_word_tokenizer,
    load_ru_geo,
    load_uk_geo,
)
from helpers.config import KSE_DIR, RU_GEO_PATH, UK_GEO_PATH

PROCESSED_PATH = KSE_DIR / "uk_geo_dataset_processed_v1.parquet"
TEST_PATH = KSE_DIR / "test.csv"
LABELING_SAMPLE_PATH = KSE_DIR / "labeling_sample.csv"
RU_NROWS = 1_000_000

In [ ]:
def entity_spans(row):
    return pd.DataFrame(
        [
            {
                "type": col.split("_")[0].upper(),
                "start": l,
                "end": r,
                "text": row["text"][l:r],
            }
            for col in MARKER_COLS
            for l, r in row[col]
        ]
    )


def entity_counts(df):
    return {
        col.split("_")[0].upper(): int(df[col].map(len).sum()) for col in MARKER_COLS
    }

## uk_geo


### Data


In [ ]:
uk_geo = load_uk_geo(UK_GEO_PATH)
train_uk_geo = uk_geo[uk_geo["split"] == "train"].reset_index(drop=True)
valid_uk_geo = uk_geo[uk_geo["split"] == "valid"].reset_index(drop=True)

print("Number of train sentences:", len(train_uk_geo))
print("Number of valid sentences:", len(valid_uk_geo))

In [ ]:
uk_geo.head()

In [ ]:
uk_geo_example = uk_geo[uk_geo[list(MARKER_COLS)].map(len).sum(axis=1) > 0].iloc[0]
print(uk_geo_example["text"])
entity_spans(uk_geo_example)

In [ ]:
uk_geo_entity_dist = entity_counts(uk_geo)

print("Entity distribution:")
for label, count in uk_geo_entity_dist.items():
    print(f"{label}: {count}")

### Vocab


In [ ]:
uk_geo_tokens = uk_geo["text"].str.split()
uk_geo_tokenizer = build_word_tokenizer(uk_geo_tokens, specials=(UNK_TOKEN, PAD_TOKEN))

tokens = uk_geo_example["text"].split()
print("Tokens:", tokens)
print("Token indices:", uk_geo_tokenizer.convert_tokens_to_ids(tokens))

In [ ]:
len(uk_geo_tokenizer)

In [ ]:
uk_geo_token_freqs = collections.Counter(tok for toks in uk_geo_tokens for tok in toks)
uk_geo_token_freqs.most_common(30)

## ru_geo


### Data


In [ ]:
ru_geo = load_ru_geo(RU_GEO_PATH, nrows=RU_NROWS)
print("Number of sentences:", len(ru_geo))

In [ ]:
ru_geo.head()

In [ ]:
ru_geo_example = ru_geo[ru_geo[list(MARKER_COLS)].map(len).sum(axis=1) > 0].iloc[0]
print(ru_geo_example["text"])
entity_spans(ru_geo_example)

In [ ]:
ru_geo_entity_dist = entity_counts(ru_geo)

print("Entity distribution:")
for label, count in ru_geo_entity_dist.items():
    print(f"{label}: {count}")

### Vocab


In [ ]:
ru_geo_tokens = ru_geo["text"].str.split()
ru_geo_tokenizer = build_word_tokenizer(ru_geo_tokens, specials=(UNK_TOKEN, PAD_TOKEN))

tokens = ru_geo_example["text"].split()
print("Tokens:", tokens)
print("Token indices:", ru_geo_tokenizer.convert_tokens_to_ids(tokens))

In [ ]:
len(ru_geo_tokenizer)

In [ ]:
ru_geo_token_freqs = collections.Counter(tok for toks in ru_geo_tokens for tok in toks)
ru_geo_token_freqs.most_common(30)

In [ ]:
for _, row in ru_geo[ru_geo["doc_id"] == 2].sort_values("sent_id").iterrows():
    print(row["sent_id"], row["text"])

In [ ]:
ru_geo_doc_sizes = ru_geo.groupby("doc_id").size()

plt.title("ru_geo Sentences per Document")
ru_geo_doc_sizes.hist(bins=100)
plt.show()

## uk_geo processed


### Data


In [ ]:
uk_geo_processed = pd.read_parquet(PROCESSED_PATH)
uk_geo_processed["tokens"] = uk_geo_processed["tokens"].map(json.loads)
uk_geo_processed["labels"] = uk_geo_processed["labels"].map(json.loads)

train_df = uk_geo_processed[uk_geo_processed["is_valid"] == 0].reset_index(drop=True)
valid_df = uk_geo_processed[uk_geo_processed["is_valid"] == 1].reset_index(drop=True)

print("Number of train sentences:", len(train_df))
print("Number of valid sentences:", len(valid_df))

In [ ]:
display(uk_geo_processed.head())
display(uk_geo.head(100))

In [ ]:
example = train_df[train_df["labels"].map(lambda ls: any(l != "O" for l in ls))].iloc[0]
pd.DataFrame({"token": example["tokens"], "label": example["labels"]})

In [ ]:
label_dist = collections.Counter(l for labels in train_df["labels"] for l in labels)

print("Unique labels:")
print(list(label_dist.keys()))

print("\nLabel distribution:")
for label, count in label_dist.items():
    print(f"{label}: {count}")

### Vocab


In [ ]:
tokenizer = build_word_tokenizer(
    pd.concat([train_df["tokens"], valid_df["tokens"]]), specials=(UNK_TOKEN, PAD_TOKEN)
)

tokens = example["tokens"]
token_indices = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token indices:", token_indices)

In [ ]:
len(tokenizer)

In [ ]:
tokenizer.convert_tokens_to_ids(["поки", "що", "покищо", "покищо#", "<unk>", "<pad>"])

In [ ]:
token_freqs = collections.Counter(
    tok for tokens in train_df["tokens"] for tok in tokens
)

# Get the 30 most common tokens
top30 = token_freqs.most_common(30)
top30

## test


### Data


In [ ]:
test = pd.read_csv(TEST_PATH).drop(columns="locations")  # locations empty in test
print("Number of texts:", len(test))

In [ ]:
test.head()

In [ ]:
print(test.loc[1, "text"])

### Vocab


In [ ]:
test_tokens = test["text"].str.split()
test_tokenizer = build_word_tokenizer(test_tokens, specials=(UNK_TOKEN, PAD_TOKEN))

tokens = test.loc[200, "text"].split()
print("Tokens:", tokens)
print("Token indices:", test_tokenizer.convert_tokens_to_ids(tokens))

In [ ]:
len(test_tokenizer)

In [ ]:
test_token_freqs = collections.Counter(tok for toks in test_tokens for tok in toks)
test_token_freqs.most_common(30)

## labeling_sample


In [ ]:
labeling_sample = pd.read_csv(LABELING_SAMPLE_PATH)
for col in ["locations", "loc_markers"]:
    labeling_sample[col] = labeling_sample[col].map(ast.literal_eval)
print("Number of texts:", len(labeling_sample))

In [ ]:
labeling_sample.head()

In [ ]:
labeling_sample_example = labeling_sample[
    labeling_sample["locations"].map(len) > 0
].iloc[0]
print(labeling_sample_example["text"])
pd.DataFrame(
    {
        "location": labeling_sample_example["locations"],
        "marker": labeling_sample_example["loc_markers"],
        "text[l:r]": [
            labeling_sample_example["text"][l:r]
            for l, r in labeling_sample_example["loc_markers"]
        ],
    }
)